# Interior Design Chatbot - RAG Version

- LLM: `Qwen/Qwen2.5-1.5B-Instruct`
- Embeddings: `sentence-transformers/all-MiniLM-L6-v2`
- Vector store: FAISS
- Framework: LangChain

Set runtime to T4 GPU on Colab before running.

## Install dependencies

In [ ]:
!pip install langchain langchain-community langchain-huggingface langchain-text-splitters \
             faiss-cpu sentence-transformers pypdf \
             transformers accelerate ipywidgets -q

## Imports and GPU check

In [ ]:
import os, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Load PDFs

If running locally, set `PDF_DIR` to the folder with the PDFs.
On Colab, upload the files or mount Google Drive.

In [ ]:
# change this to wherever your PDFs are
PDF_DIR = Path("Techniques and methods")

# if on colab, uncomment below to upload instead
# from google.colab import files
# uploaded = files.upload()
# PDF_DIR = Path("/content")

pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs:")
for p in pdf_files:
    print(f"  {p.name}")

In [ ]:
# load all PDFs
all_docs = []
for pdf_path in pdf_files:
    try:
        loader = PyPDFLoader(str(pdf_path))
        docs = loader.load()
        for doc in docs:
            doc.metadata["source"] = pdf_path.name
        all_docs.extend(docs)
        print(f"  {pdf_path.name} - {len(docs)} pages")
    except Exception as e:
        print(f"  FAILED: {pdf_path.name} - {e}")

print(f"\nTotal pages: {len(all_docs)}")

## Chunk text and build FAISS index

In [ ]:
# split into smaller chunks for retrieval
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(all_docs)
print(f"Total chunks: {len(chunks)}")
print(f"\nSample chunk:")
print(chunks[0].page_content[:300])
print(f"Source: {chunks[0].metadata['source']}")

In [ ]:
FAISS_DIR = "interior_faiss_index"

print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": DEVICE},
)

# build FAISS index from scratch, or load if already exists
if os.path.exists(FAISS_DIR):
    print("Loading existing FAISS index...")
    db = FAISS.load_local(FAISS_DIR, embeddings, allow_dangerous_deserialization=True)
else:
    print("Building FAISS index (takes about a minute)...")
    db = FAISS.from_documents(chunks, embeddings)
    db.save_local(FAISS_DIR)

print(f"FAISS index ready - {db.index.ntotal} vectors")

## Load the LLM

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
    device_map="auto",
)
model.eval()

# wrap in a langchain-compatible pipeline
hf_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.3,       # keep it low for more factual RAG answers
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False,
)

llm = HuggingFacePipeline(pipeline=hf_pipe)

if DEVICE == "cuda":
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Model loaded - VRAM: {used:.1f} / {total:.1f} GB")
else:
    print("Model loaded on CPU")

## Build the RAG chain

In [ ]:
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},   # retrieve top 4 chunks
)

RAG_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are an expert interior design consultant. "
        "Use ONLY the context below to answer the question. "
        "If the answer is not in the context, say 'I don't have enough information in my documents to answer that.'\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\n\n"
        "Answer:"
    ),
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("RAG chain ready")

## Quick test

In [ ]:
def ask(question):
    """Ask a question and return the answer + source docs."""
    docs = retriever.invoke(question)
    answer = rag_chain.invoke(question).strip()
    sources = list({doc.metadata["source"] for doc in docs})
    return {"answer": answer, "sources": sources}

# test it
test_q = "What are the key principles of interior design?"
out = ask(test_q)
print(f"Q: {test_q}")
print(f"A: {out['answer']}")
print(f"\nSources: {out['sources']}")

## Chat UI (optional)

Some questions to try:
- "What are the key principles of interior design?"
- "How does lighting affect the mood of a room?"
- "What design styles are covered in the documents?"
- "What materials are recommended for flooring?"
- "How should I brief an interior designer?"

In [ ]:
# sample VLM output to test the analysis pipeline
vlm_output = {
  "image_path": "uploads/365Cupperserangoonroad_11.jpg",
  "user_prompt": "I want a Japanese themed house",
  "vision_output": {
    "objects": ["desk","chair","bed","window","track lighting","shelving unit","plant"],
    "room_type": "bedroom",
    "scene_style": "modern",
    "object_notes": {
      "desk": "long wooden desk",
      "chair": "simple beige chair",
      "bed": "white bed with striped blanket",
      "window": "large window with blinds",
      "track lighting": "black track lighting fixtures",
      "shelving unit": "wooden shelving unit on wall",
      "plant": "potted plant on windowsill"
    }
  },
  "edit_plan": {
    "global_style": "Japanese themed house",
    "edits": [
      {"object": "desk",          "description": "a traditional Japanese low table with a natural wood finish"},
      {"object": "chair",         "description": "a zabuton cushion for sitting on the floor"},
      {"object": "bed",           "description": "a futon mattress with a woven cover"},
      {"object": "window",        "description": "paper shoji screens instead of blinds"},
      {"object": "track lighting","description": "paper lanterns hanging from the ceiling"},
      {"object": "shelving unit", "description": "a built-in bamboo shelf unit"},
      {"object": "plant",         "description": "a bonsai tree in a traditional pot"}
    ]
  }
}

print(f"User prompt: {vlm_output['user_prompt']}")
print(f"Room type: {vlm_output['vision_output']['room_type']}")
print(f"Edits: {len(vlm_output['edit_plan']['edits'])}")


# build the query from VLM output and run through RAG
vout  = vlm_output["vision_output"]
eplan = vlm_output["edit_plan"]

notes = "\n".join(f"  - {obj}: {desc}" for obj, desc in vout["object_notes"].items())
edits = "\n".join(f"  - {e['object']}: {e['description']}" for e in eplan["edits"])

query = (
    f"A {vout['room_type']} with a {vout['scene_style']} style contains these objects:\n{notes}\n\n"
    f"The user wants: \"{vlm_output['user_prompt']}\"\n\n"
    f"A vision model proposed these edits:\n{edits}\n\n"
    f"You are a specialist in {eplan['global_style']} interior design. "
    f"For EACH object listed, provide a detailed review using this format:\n\n"
    f"OBJECT: [name]\n"
    f"  Current: [what it is now]\n"
    f"  Proposed: [what was suggested]\n"
    f"  Authenticity: [is it true to {eplan['global_style']} style? why?]\n"
    f"  Improved suggestion: [more specific alternative]\n"
    f"  Why it works: [brief reason grounded in {eplan['global_style']} design principles]\n\n"
    f"After reviewing all objects, add:\n"
    f"MISSING ELEMENTS: List 3-5 additional objects, materials or textures not yet mentioned "
    f"that are essential to complete an authentic {eplan['global_style']} room.\n\n"
    f"Be specific - avoid vague terms like 'natural materials', name the exact material."
)

out = ask(query)

print("=" * 50)
print(f"Style: {eplan['global_style']}")
print(f"Room: {vout['room_type']}")
print("=" * 50)
print()
print(out["answer"])
print(f"\nSources: {', '.join(out['sources'])}")

In [ ]:
import time, re, statistics, json
from datetime import datetime
from pathlib import Path

SYSTEM_PROMPT = (
    "You are a knowledgeable and friendly interior design consultant. "
    "Give practical, actionable advice about space planning, colour palettes, "
    "lighting, furniture arrangement, and design styles."
)

def build_analysis_prompt(vlm):
    vout  = vlm["vision_output"]
    eplan = vlm["edit_plan"]
    notes = "\n".join(f"  - {obj}: {desc}" for obj, desc in vout["object_notes"].items())
    edits = "\n".join(f"  - {e['object']}: {e['description']}" for e in eplan["edits"])
    return (
        f"A {vout['room_type']} with a {vout['scene_style']} style contains:\n{notes}\n\n"
        f"The user wants: \"{vlm['user_prompt']}\"\n\n"
        f"A vision model proposed these edits:\n{edits}\n\n"
        f"You are a specialist in {eplan['global_style']} interior design. "
        f"For EACH object listed, respond using this format:\n\n"
        f"OBJECT: [name]\n"
        f"  Material: [exact material, e.g. solid oak, brushed steel]\n"
        f"  Texture: [exact texture, e.g. matte, hand-scraped, woven]\n"
        f"  Colour: [exact colour tone, e.g. warm ivory #F5F0E8]\n"
        f"  Improved suggestion: [specific alternative]\n"
        f"  Why it works: [reason tied to {eplan['global_style']} design principles]\n\n"
        f"MISSING ELEMENTS: List 3-5 essential items not yet mentioned.\n\n"
        f"Name exact materials and textures. Avoid vague terms."
    )

# base room - same for all styles
BASE_ROOM = {
    "objects": ["desk","chair","bed","window","track lighting","shelving unit","plant"],
    "room_type": "bedroom",
    "scene_style": "modern",
    "object_notes": {
        "desk":           "long wooden desk",
        "chair":          "simple beige chair",
        "bed":            "white bed with striped blanket",
        "window":         "large window with blinds",
        "track lighting": "black track lighting fixtures",
        "shelving unit":  "wooden shelving unit on wall",
        "plant":          "potted plant on windowsill",
    }
}

# styles to benchmark
DESIGNS = {
    "japanese": {
        "prompt": "I want a Japanese themed bedroom",
        "edits": [
            {"object": "desk",           "description": "a traditional low chabudai table with a natural wood finish"},
            {"object": "chair",          "description": "a zabuton floor cushion for seated work"},
            {"object": "bed",            "description": "a futon mattress with a woven indigo cover"},
            {"object": "window",         "description": "paper shoji screens replacing the blinds"},
            {"object": "track lighting", "description": "paper lanterns suspended from the ceiling"},
            {"object": "shelving unit",  "description": "a built-in bamboo shelf unit"},
            {"object": "plant",          "description": "a bonsai tree in a glazed ceramic pot"},
        ]
    },
    "victorian": {
        "prompt": "I want a Victorian style bedroom",
        "edits": [
            {"object": "desk",           "description": "an ornate writing desk with carved dark mahogany legs"},
            {"object": "chair",          "description": "a tufted wingback chair in deep burgundy velvet"},
            {"object": "bed",            "description": "a four-poster bed with heavy damask drapes"},
            {"object": "window",         "description": "floor-length velvet curtains with brass tiebacks"},
            {"object": "track lighting", "description": "a brass chandelier with frosted glass globe shades"},
            {"object": "shelving unit",  "description": "a dark walnut bookcase with glass doors and brass hardware"},
            {"object": "plant",          "description": "a large fern in a brass jardiniere stand"},
        ]
    },
    "modern": {
        "prompt": "I want a modern minimalist bedroom",
        "edits": [
            {"object": "desk",           "description": "a wall-mounted floating desk in white lacquer"},
            {"object": "chair",          "description": "a moulded shell chair on slim hairpin legs"},
            {"object": "bed",            "description": "a low platform bed with an upholstered light grey headboard"},
            {"object": "window",         "description": "floor-to-ceiling roller blinds in warm white"},
            {"object": "track lighting", "description": "recessed LED spotlights flush to the ceiling"},
            {"object": "shelving unit",  "description": "a modular open shelving system in white powder-coated steel"},
            {"object": "plant",          "description": "a single sculptural cactus in a concrete planter"},
        ]
    },
    "scandinavian": {
        "prompt": "I want a Scandinavian hygge bedroom",
        "edits": [
            {"object": "desk",           "description": "a light birch desk with tapered solid wood legs"},
            {"object": "chair",          "description": "a sheepskin-draped solid ash stool"},
            {"object": "bed",            "description": "a low ash frame with a white cotton duvet and knit throw"},
            {"object": "window",         "description": "sheer undyed linen curtains that pool softly on the floor"},
            {"object": "track lighting", "description": "a white paper pendant lamp on a fabric cord"},
            {"object": "shelving unit",  "description": "a floating light pine shelf with minimal objects"},
            {"object": "plant",          "description": "eucalyptus stems in a simple matte ceramic vase"},
        ]
    },
    "industrial": {
        "prompt": "I want an industrial loft bedroom",
        "edits": [
            {"object": "desk",           "description": "a reclaimed oak plank desk on black iron pipe legs"},
            {"object": "chair",          "description": "a metal factory stool with a worn leather seat pad"},
            {"object": "bed",            "description": "a black powder-coated steel bed frame with visible hex bolts"},
            {"object": "window",         "description": "steel-framed windows left bare to maximise light"},
            {"object": "track lighting", "description": "bare Edison bulb pendants on black iron conduit pipe"},
            {"object": "shelving unit",  "description": "iron pipe brackets holding raw-edge reclaimed wood planks"},
            {"object": "plant",          "description": "a tall snake plant in a galvanised steel bucket"},
        ]
    },
    "bohemian": {
        "prompt": "I want a bohemian eclectic bedroom",
        "edits": [
            {"object": "desk",           "description": "a distressed pine desk with a macrame-trimmed drawer pull"},
            {"object": "chair",          "description": "a rattan hanging egg chair with layered kilim cushions"},
            {"object": "bed",            "description": "a low divan layered with patterned textiles and throw pillows"},
            {"object": "window",         "description": "sheer embroidered muslin curtains with tassel trim"},
            {"object": "track lighting", "description": "warm string fairy lights and a woven rattan pendant"},
            {"object": "shelving unit",  "description": "mismatched reclaimed wood floating shelves with trailing plants"},
            {"object": "plant",          "description": "trailing pothos in a hanging macrame plant holder"},
        ]
    },
}

# scoring

GENERIC_TERMS = [
    "natural wood", "soft fabric", "traditional style", "traditional material",
    "wooden material", "nice", "elegant", "classic",
]
SPECIFIC_PATTERNS = [
    r"\b(hinoki|sugi|cedar|cypress|paulownia|kiri|persimmon|mahogany|walnut|birch|ash|oak|pine|teak)\b",
    r"\b(ramie|habotai|asa|hemp|linen|washi|kozo|velvet|damask|muslin|kilim|cotton|sheepskin)\b",
    r"\b(urushi|lacquer|tung.oil|beeswax|shou.sugi|charred|powder.coated|galvanised)\b",
    r"\b(shigaraki|bizen|stoneware|brass|iron|steel)\b",
    r"\d+\s*(cm|mm|inches?)",
    r"#[0-9a-fA-F]{6}",
]

def score_completeness(output, vlm):
    objects = [e["object"].lower() for e in vlm["edit_plan"]["edits"]]
    covered = [o for o in objects if o in output.lower()]
    return {
        "score": round(len(covered) / len(objects), 3),
        "covered": covered,
        "missing": [o for o in objects if o not in covered],
    }

def score_specificity(output):
    text = output.lower()
    generic_hits = sum(1 for t in GENERIC_TERMS if t in text)
    specific_hits, found = 0, []
    for pat in SPECIFIC_PATTERNS:
        matches = re.findall(pat, text)
        specific_hits += len(matches)
        found.extend(matches)
    total = specific_hits + generic_hits
    return {
        "score": round(specific_hits / total, 3) if total > 0 else 0.0,
        "specific_count": specific_hits,
        "generic_count": generic_hits,
        "found": list(set(found))[:8],
    }

def run_once(vlm):
    prompt = build_analysis_prompt(vlm)
    t0 = time.time()
    result = ask(prompt)
    elapsed = time.time() - t0
    decoded = result["answer"]
    n_tok = len(decoded.split())
    return decoded, elapsed, n_tok

# run benchmark
N_RUNS = 3
all_results = {}

for style, cfg in DESIGNS.items():
    vlm = {
        "image_path":    "uploads/test_room.jpg",
        "user_prompt":   cfg["prompt"],
        "vision_output": BASE_ROOM,
        "edit_plan":     {"global_style": f"{style} style", "edits": cfg["edits"]},
    }

    print(f"\n{'='*50}")
    print(f"  {style.upper()} ({N_RUNS} runs)")
    print(f"{'='*50}")

    runs = []
    for i in range(N_RUNS):
        print(f"  run {i+1}/{N_RUNS} ...", end=" ", flush=True)
        out, elapsed, n_tok = run_once(vlm)
        c = score_completeness(out, vlm)
        s = score_specificity(out)
        print(f"{elapsed:.1f}s | comp={c['score']*100:.0f}% | spec={s['score']*100:.0f}%")
        runs.append({
            "run":          i + 1,
            "output":       out,
            "completeness": c,
            "specificity":  s,
            "speed":        {"elapsed_s": round(elapsed, 2), "tokens": n_tok,
                             "tok_per_s": round(n_tok / elapsed, 1)},
        })

    c_vals = [r["completeness"]["score"] for r in runs]
    s_vals = [r["specificity"]["score"]  for r in runs]
    t_vals = [r["speed"]["elapsed_s"]    for r in runs]

    all_results[style] = {
        "runs": runs,
        "summary": {
            "completeness_mean":  round(statistics.mean(c_vals), 3),
            "completeness_stdev": round(statistics.stdev(c_vals), 3),
            "specificity_mean":   round(statistics.mean(s_vals), 3),
            "specificity_stdev":  round(statistics.stdev(s_vals), 3),
            "speed_mean_s":       round(statistics.mean(t_vals), 2),
            "speed_stdev_s":      round(statistics.stdev(t_vals), 2),
        }
    }

# save results
out_path = Path("benchmark_results.json")
out_path.write_text(json.dumps({
    "timestamp":        datetime.now().isoformat(),
    "model":            MODEL_ID,
    "n_runs_per_style": N_RUNS,
    "results":          all_results,
}, indent=2))
print(f"\nResults saved to {out_path}")

# markdown report
md_lines = [
    f"# Benchmark Report\n",
    f"**Model:** {MODEL_ID}  |  **Runs per style:** {N_RUNS}  |  "
    f"**Date:** {datetime.now().strftime('%Y-%m-%d %H:%M')}\n",
    "| Style | Comp mean | Comp +/- | Spec mean | Spec +/- | Speed (s) | Speed +/- |",
    "|---|---|---|---|---|---|---|",
]
for style, data in all_results.items():
    s = data["summary"]
    md_lines.append(
        f"| {style} | {s['completeness_mean']:.2f} | {s['completeness_stdev']:.3f} "
        f"| {s['specificity_mean']:.2f} | {s['specificity_stdev']:.3f} "
        f"| {s['speed_mean_s']:.1f} | {s['speed_stdev_s']:.2f} |"
    )

md_lines += ["\n## Outputs\n"]
for style, data in all_results.items():
    md_lines.append(f"### {style.title()}")
    for r in data["runs"]:
        md_lines.append(f"\n**Run {r['run']}** - "
                        f"comp={r['completeness']['score']*100:.0f}% | "
                        f"spec={r['specificity']['score']*100:.0f}% | "
                        f"{r['speed']['elapsed_s']}s\n")
        md_lines.append(r["output"])
        md_lines.append("\n---")

Path("benchmark_report.md").write_text("\n".join(md_lines))
print("Report saved to benchmark_report.md")

# summary table
print(f"\n{'Style':<15} {'Comp':>6} {'+-':>6} {'Spec':>6} {'+-':>6} {'Speed':>8} {'+-':>6}")
print("-" * 55)
for style, data in all_results.items():
    s = data["summary"]
    print(f"{style:<15} {s['completeness_mean']:>6.2f} {s['completeness_stdev']:>6.3f} "
          f"{s['specificity_mean']:>6.2f} {s['specificity_stdev']:>6.3f} "
          f"{s['speed_mean_s']:>8.1f} {s['speed_stdev_s']:>6.2f}")